# logistic regression Model 

This notebook trains and evaluates a text-classification model for predicting the article category. It uses TF-IDF to represent article text numerically and logistic regression to predict the class probabilities.

The cells are arranged as a complete experiment: first the environment and data are prepared, then several model settings are compared, and finally the selected configuration is evaluated and used to create error-analysis files and a submission.



## 1. Setup

This section identifies the experiment and prepares the Python environment. The first code cell records the contributor and model name, the next cell makes the repository modules available both locally and in Google Colab, and the imports cell loads the libraries and project helpers used throughout the notebook.

In [ ]:
MEMBER = "vestine umukundwa"
MODEL_NAME = "logistic_regression"

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone https://github.com/michael-alu/nlp_language_technologies.git
    %cd nlp_language_technologies
    %pip install -q -r requirements.txt

if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src import config
from src import data_loading
from src import evaluation
from src import plots
from src.experiment_log import log_experiment
from src.reproducibility import FINAL_RUN_SEEDS, set_seed

## 2. Load the data

This cell loads the prepared training, validation, test, and competition datasets. It also converts category names into the numeric class IDs required by scikit-learn, combines training and validation data for cross-validation, and prints basic class and dataset-size checks so that loading problems or class imbalance are visible early.

In [ ]:
train = data_loading.load_train()
validation = data_loading.load_validation()
test = data_loading.load_test()
zindi_test = data_loading.load_zindi_test()

train_ids = data_loading.labels_to_ids(train["category"])
validation_ids = data_loading.labels_to_ids(validation["category"])
test_ids = data_loading.labels_to_ids(test["category"])

# Used by cross-validation
train_and_validation = data_loading.load_train_and_validation()

print("Labels:", config.LABELS)
print("Sizes:", len(train), len(validation), len(test))
print(train["category"].value_counts())

## 3. The model

`build_model()` makes a TF-IDF + logistic regression pipeline. Each experiment just sets a `settings` dictionary; anything it does not set uses a sensible default. `run_experiment()` scores the settings on the validation split and with cross-validation, and logs both.

In [ ]:
settings = {}  # Settings for the current experiment


def build_model():
    # Convert text into TF-IDF features
    vectorizer = TfidfVectorizer(
        analyzer=settings.get("analyzer", "word"),
        ngram_range=settings.get("ngram_range", (1, 1)),
        min_df=2,          # Ignore very rare words
        max_df=0.9,        # Ignore very common words
        sublinear_tf=True,
        lowercase=True,
    )

    # Create the classifier
    classifier = LogisticRegression(
        C=settings.get("C", 1.0),
        class_weight=settings.get("class_weight", None),
        max_iter=1000,
        solver="lbfgs",
        random_state=config.RANDOM_SEED,
    )

    # Combine TF-IDF and classifier
    return Pipeline([
        ("tfidf", vectorizer),
        ("classifier", classifier)
    ])


def run_experiment(name, notes):
    # Train and evaluate on validation data
    set_seed(config.RANDOM_SEED)
    model = build_model()
    model.fit(train["content"], train_ids)

    # Get predictions
    val_probs = model.predict_proba(validation["content"])

    # Calculate scores
    val_scores = evaluation.compute_scores(validation_ids, val_probs)
    evaluation.print_classification_report(validation_ids, val_probs)

    # Save validation results
    log_experiment(
        MEMBER, MODEL_NAME, name, "validation",
        settings, val_scores, notes
    )

    # Run cross-validation
    cv_scores = cross_validate(build_model, train_and_validation)

    # Save cross-validation results
    log_experiment(
        MEMBER, MODEL_NAME, name, "cross_validation",
        settings, cv_scores, notes
    )

    return model, val_probs, val_scores, cv_scores

## 4. Experiments

Compare settings using the **cross-validated** macro-F1, because the validation split has only 2 `burudani` articles, so one mistake changes the single-split score a lot.

### 4.1 Word unigrams (baseline) - how far do single words get?

This is the reference model. It represents each article using individual words only, so later configurations can be judged by whether they improve on this simple baseline.

In [ ]:
from src.cross_validation import cross_validate
settings = {"analyzer": "word", "ngram_range": (1, 1)}
model, val_probs, val_scores, cv_scores = run_experiment("word_unigrams", "Single words only (baseline).")
val_scores

### 4.2 Add bigrams - do pairs like "Ligi Kuu" help?

This cell adds two-word sequences to the single-word features. Bigrams can preserve short phrases and word order, which may be more informative than either word by itself. Compare its cross-validated macro-F1 with the unigram baseline.

In [ ]:
settings = {"analyzer": "word", "ngram_range": (1, 2)}
model, val_probs, val_scores, cv_scores = run_experiment("word_bigrams", "Words + word pairs (bigrams).")
val_scores

### 4.3 Character n-grams - do word pieces handle Swahili prefixes/suffixes better?

This experiment represents short character sequences inside word boundaries instead of complete words. Character n-grams can share information across related word forms, spelling variations, and prefixes or suffixes. The output shows whether that robustness improves the validation and cross-validation scores.

In [ ]:
settings = {"analyzer": "char_wb", "ngram_range": (2, 5)}
model, val_probs, val_scores, cv_scores = run_experiment("char_ngrams", "Character n-grams (2-5).")
val_scores

### 4.4 Balanced class weights - does weighting help `kimataifa` and `burudani`?

This run keeps the word-bigram representation but gives more influence to underrepresented classes during training. This tests whether correcting the class imbalance improves macro-F1, especially for the smaller categories, without relying only on overall accuracy.

In [ ]:
settings = {"analyzer": "word", "ngram_range": (1, 2), "class_weight": "balanced"}
model, val_probs, val_scores, cv_scores = run_experiment("word_bigrams_balanced", "Bigrams + class_weight=balanced.")
val_scores

### 4.5 Tune C - how sensitive is the model to overfitting?

`C` controls the strength of logistic-regression regularization: smaller values constrain the model more, while larger values allow it to fit the training features more closely. The loop evaluates three values using the same balanced word-bigram pipeline and prints each cross-validated macro-F1 for comparison.

In [ ]:
for c in [0.1, 1.0, 10.0]:
    settings = {"analyzer": "word", "ngram_range": (1, 2), "class_weight": "balanced", "C": c}
    print(f"\n========== C = {c} ==========")
    model, val_probs, val_scores, cv_scores = run_experiment(f"tune_C_{c}", f"Tuning C = {c}.")
    print(f"C = {c}  ->  CV macro-F1 = {cv_scores['macro_f1']}")

### 4.6 Character n-grams + balanced class weights - do the best features also work best with weights?

*Added by Michael after reviewing the experiments above.*

Before weighting, character n-grams (4.3) had the best cross-validated macro-F1 and log loss of the three feature types, while bigrams (4.2) had the worst. Experiments 4.4 and 4.5 added class weights and tuned `C` on bigrams, so this run checks whether class weights work even better on top of the stronger character features.

In [ ]:
settings = {"analyzer": "char_wb", "ngram_range": (2, 5), "class_weight": "balanced"}
model, val_probs, val_scores, cv_scores = run_experiment("char_ngrams_balanced", "Character n-grams (2-5) + class_weight=balanced.")
val_scores

### 4.7 Tune C for character n-grams + balanced class weights

*Added by Michael.*

The same `C` values as in 4.5, now on the character setup. `C = 1.0` is the default and was already tested in 4.6, so the loop only adds `C = 0.1` and `C = 10.0`.

In [ ]:
for c in [0.1, 10.0]:
    settings = {"analyzer": "char_wb", "ngram_range": (2, 5), "class_weight": "balanced", "C": c}
    print(f"\n========== C = {c} ==========")
    model, val_probs, val_scores, cv_scores = run_experiment(f"char_balanced_C_{c}", f"Character n-grams + balanced, C = {c}.")
    print(f"C = {c}  ->  CV macro-F1 = {cv_scores['macro_f1']}")

### Best configuration - inspect the errors

After comparing the experiment logs, set `settings` to the configuration with the strongest and most reliable cross-validated macro-F1. This cell reruns that choice, displays its validation scores, and draws a confusion matrix so we can see which categories are being confused before using the model on the test set.

**Decision (added by Michael): character n-grams (2-5), balanced class weights, `C = 1.0`.**

- Class weights were the biggest single improvement (4.4). Without them, the model never predicted `kimataifa` or `burudani`.
- With class weights, character n-grams beat word bigrams clearly (cross-validated macro-F1 0.633 vs 0.563, log loss 0.474 vs 0.534). Character pieces can match different forms of the same Swahili word (prefixes and suffixes), which whole-word features treat as unrelated words.
- For character n-grams, `C = 0.1` had about the same macro-F1 (0.636), but its log loss was almost twice as bad (0.905), because strong regularization makes the probabilities too uncertain. `C = 10.0` had the best log loss (0.355) but a lower macro-F1 (0.623), and its `kimataifa` F1 dropped from 0.45 to 0.29. `C = 1.0` is the best balance between the two metrics.

In [ ]:
settings = {"analyzer": "char_wb", "ngram_range": (2, 5), "class_weight": "balanced", "C": 1.0}  # <-- set to your best
model, val_probs, val_scores, cv_scores = run_experiment("best_config", "Best settings from the experiments above.")

plots.plot_confusion_matrix(validation_ids, val_probs, MODEL_NAME, "validation")
val_scores

In [ ]:
from src.experiment_log import EXPERIMENT_LOGS_DIRECTORY

def show_experiment_log():
    log = pd.read_csv(EXPERIMENT_LOGS_DIRECTORY / f"{MODEL_NAME}.csv")
    columns_to_show = ["experiment", "split", "macro_f1", "log_loss", "accuracy", "f1_kimataifa", "f1_burudani"]
    return log[columns_to_show].sort_values("macro_f1", ascending=False)

show_experiment_log()

## 5. Final test (run once)

In [ ]:
# Best model settings
settings = {
    "analyzer": "char_wb",
    "ngram_range": (2, 5),       # Use character pieces of 2 to 5 letters
    "class_weight": "balanced",  # Handle imbalanced classes
    "C": 1.0
}

# Set random seed
set_seed(config.RANDOM_SEED)

# Build the model
model = build_model()

# Train on the training data
model.fit(train["content"], train_ids)

# Predict probabilities on test data
test_probs = model.predict_proba(test["content"])

# Calculate test scores
test_scores = evaluation.compute_scores(test_ids, test_probs)

# Show classification results
evaluation.print_classification_report(test_ids, test_probs)
 
# Save the final results
log_experiment(
    MEMBER,
    MODEL_NAME,
    "final",
    "test",
    settings,
    test_scores,
    "Final model on the test set."
)

# Display the scores
test_scores

In [ ]:
plots.plot_confusion_matrix(test_ids, test_probs, MODEL_NAME, "test")
plots.plot_roc_curves(test_ids, test_probs, MODEL_NAME, "test")

These plots provide a visual check of the final test predictions: the confusion matrix summarizes category-level mistakes, while the ROC curves show how well the predicted probabilities separate each class.

In [ ]:
# Saves results/misclassified/logistic_regression.csv for the error analysis
evaluation.save_misclassified_examples(test, test_ids, test_probs, MODEL_NAME)

In [ ]:
zindi_probs = model.predict_proba(zindi_test["content"])
evaluation.save_zindi_submission(zindi_test, zindi_probs, MODEL_NAME)